This model has already the best parameters found in Bayesian Optimization (Leaderboard score 0.728 Macro F1)

In [ ]:

# Libraries and Frameworks
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
# Paths 

DEV_IN_PATH  = "../../data/processed/development_processed.csv"
EVAL_IN_PATH = "../../data/processed/evaluation_processed.csv"

SUB_OUT = "../../data/submission/submission_baseline.csv"




In [ ]:
# Parameters 
WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.8782583211530898
C_VALUE     = 0.64491922705094

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]


In [ ]:
# Load data and Features 

df_dev  = pd.read_csv(DEV_IN_PATH)
df_eval = pd.read_csv(EVAL_IN_PATH)

FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]


In [ ]:
# Define Model (Lineaer Regression with TF-IDF and Scaling)

def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])

In [ ]:
# Train and Predict

model = make_model()
model.fit(X_dev, y_dev)

pred = model.predict(X_eval)

In [ ]:
# Submission

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": pred.astype(int)
})

submission.to_csv(SUB_OUT, index=False)
print("Saved baseline submission to:", SUB_OUT)